# 16.2 防御方法 (Defense Methods)

防御方法使模型对抗对抗性攻击，提升鲁棒性和安全性。

本节涵盖：
- 输入过滤与净化
- 输出审查与安全护栏
- 对抗训练
- 安全对齐（RLHF/Constitutional AI）
- 深度防御策略
- 红队测试与安全评估

## 1. 输入过滤与净化

**输入过滤**是第一道防线，在恶意输入到达模型前进行拦截。

### 过滤方法
- **规则过滤**：关键词黑名单、正则表达式匹配
- **ML过滤**：训练分类器检测注入/越狱模式
- **困惑度过滤**：对抗样本通常具有异常高/低的困惑度
- **输入净化**：移除/转义特殊字符，规范化输入格式

### 局限性
- 规则过滤容易被绕过（同义词替换、编码）
- ML过滤存在误报（正常输入被拦截）
- 困惑度过滤对语义级攻击无效
- **核心原则**：过滤是必要但不充分的，需要与其他防御组合

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class InputFilter(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.injection_detector = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1), nn.Sigmoid()
        )
        self.perplexity_estimator = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Softplus()
        )
        self.ppl_threshold = 50.0
        self.injection_threshold = 0.5

    def forward(self, x):
        injection_risk = self.injection_detector(x)
        estimated_ppl = self.perplexity_estimator(x)
        is_safe = (injection_risk < self.injection_threshold) & (estimated_ppl < self.ppl_threshold)
        return is_safe, injection_risk, estimated_ppl

class InputSanitizer(nn.Module):
    def __init__(self, d_model=128, n_layers=2):
        super().__init__()
        layers = []
        for _ in range(n_layers):
            layers.extend([nn.Linear(d_model, d_model), nn.LayerNorm(d_model), nn.ReLU()])
        self.sanitize_net = nn.Sequential(*layers, nn.Linear(d_model, d_model))
        self.gate = nn.Sequential(nn.Linear(d_model * 2, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())

    def forward(self, system_embed, user_embed):
        sanitized = self.sanitize_net(user_embed)
        gate_input = torch.cat([system_embed, sanitized], dim=-1)
        gate_value = self.gate(gate_input)
        clean_embed = system_embed + gate_value * sanitized
        return clean_embed, gate_value

input_filter = InputFilter(d_model=128)
sanitizer = InputSanitizer(d_model=128)

system = torch.randn(1, 128)
normal_input = torch.randn(1, 128) - 0.3
attack_input = torch.randn(1, 128) + 0.8
high_ppl_input = torch.randn(1, 128) * 3.0

print('=== Input Filtering ===')
for name, inp in [('Normal', normal_input), ('Attack', attack_input), ('High PPL', high_ppl_input)]:
    safe, risk, ppl = input_filter(inp)
    print(f'{name}: safe={safe.item()}, injection_risk={risk.item():.3f}, ppl={ppl.item():.1f}')

print(f'\n=== Input Sanitization ===')
for name, inp in [('Normal', normal_input), ('Attack', attack_input)]:
    clean, gate = sanitizer(system, inp)
    print(f'{name}: gate_value={gate.item():.3f} (lower = less user influence)')

print(f'\nKey: Multi-signal filtering (injection risk + perplexity) is more robust than single-signal.')
print(f'Sanitization gates user input based on alignment with system instructions.')

## 2. 输出审查与安全护栏

**输出审查**在模型生成后检查内容安全性，是最后一道防线。

### 审查方法
- **毒性检测**：检测仇恨言论、暴力、色情等有害内容
- **PII检测**：检测个人身份信息泄露（姓名、电话、邮箱）
- **事实性检查**：验证输出是否与提供的上下文一致
- **安全分类器**：多标签安全分类

### 安全护栏 (Guardrails)
- **NeMo Guardrails**：NVIDIA开源的LLM安全框架
- **Llama Guard**：Meta的LLM安全分类器
- **Guardrails AI**：基于RAIL规范的输入输出验证

### 产业实践
OpenAI、Anthropic等均采用输入过滤+输出审查的双重机制。

In [ ]:
class OutputAuditor(nn.Module):
    def __init__(self, d_model=128, n_safety_categories=4):
        super().__init__()
        self.n_categories = n_safety_categories
        self.safety_classifier = nn.Sequential(
            nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, n_safety_categories), nn.Sigmoid()
        )
        self.pii_detector = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )
        self.faithfulness_checker = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        self.category_names = ['violence', 'hate_speech', 'sexual_content', 'self_harm']

    def forward(self, output_embed, context_embed=None):
        safety_scores = self.safety_classifier(output_embed)
        pii_risk = self.pii_detector(output_embed)
        faithfulness = None
        if context_embed is not None:
            faithfulness = self.faithfulness_checker(torch.cat([output_embed, context_embed], dim=-1))
        return safety_scores, pii_risk, faithfulness

    def is_safe(self, output_embed, context_embed=None, thresholds=None):
        if thresholds is None:
            thresholds = {'safety': 0.5, 'pii': 0.5, 'faithfulness': 0.3}
        safety_scores, pii_risk, faithfulness = self.forward(output_embed, context_embed)
        unsafe_categories = []
        for i, score in enumerate(safety_scores[0]):
            if score.item() > thresholds['safety']:
                unsafe_categories.append(self.category_names[i])
        if pii_risk.item() > thresholds['pii']:
            unsafe_categories.append('pii_leak')
        if faithfulness is not None and faithfulness.item() < thresholds['faithfulness']:
            unsafe_categories.append('unfaithful')
        return len(unsafe_categories) == 0, unsafe_categories

class SafetyGuardrail:
    def __init__(self, d_model=128):
        self.input_filter = InputFilter(d_model)
        self.output_auditor = OutputAuditor(d_model)
        self.sanitizer = InputSanitizer(d_model)
        self.blocked_count = 0
        self.audit_log = []

    def process(self, system_embed, user_embed, model_output_embed, context_embed=None):
        input_safe, inj_risk, ppl = self.input_filter(user_embed)
        if not input_safe.item():
            self.blocked_count += 1
            self.audit_log.append({'stage': 'input', 'reason': f'injection_risk={inj_risk.item():.3f}'})
            return None, 'Input blocked by safety filter'
        clean_embed, gate = self.sanitizer(system_embed, user_embed)
        output_safe, violations = self.output_auditor.is_safe(model_output_embed, context_embed)
        if not output_safe:
            self.blocked_count += 1
            self.audit_log.append({'stage': 'output', 'reason': f'violations={violations}'})
            return None, f'Output blocked: {violations}'
        return model_output_embed, 'Approved'

auditor = OutputAuditor(d_model=128)
guardrail = SafetyGuardrail(d_model=128)

normal_output = torch.randn(1, 128) - 0.3
toxic_output = torch.randn(1, 128) + 0.8
context = torch.randn(1, 128)

print('=== Output Auditing ===')
for name, out in [('Normal', normal_output), ('Toxic', toxic_output)]:
    safety, pii, faith = auditor(out, context)
    safe, violations = auditor.is_safe(out, context)
    print(f'{name}: safe={safe}, violations={violations}')
    print(f'  Safety scores: {[f"{s:.3f}" for s in safety[0].tolist()]}')
    print(f'  PII risk: {pii.item():.3f}, Faithfulness: {faith.item():.3f}')

print(f'\n=== Full Guardrail Pipeline ===')
system = torch.randn(1, 128)
user = torch.randn(1, 128)
model_out = torch.randn(1, 128)
result, status = guardrail.process(system, user, model_out, context)
print(f'Result: {status}')
print(f'Blocked count: {guardrail.blocked_count}')

print(f'\nKey: Defense-in-depth combines input filtering, sanitization, and output auditing.')
print(f'Each layer catches different types of threats.')

## 3. 对抗训练

**对抗训练**将对抗样本加入训练集，使模型学习抵抗攻击。

### 核心思想
$$\min_\theta \mathbb{E}_{(x,y) \sim D}[\max_{\delta \in S} L(\theta, x+\delta, y)]$$

内层最大化：找到最坏扰动；外层最小化：在扰动上优化模型。

### 方法对比
| 方法 | 攻击方式 | 训练成本 | 鲁棒性 |
|------|---------|---------|--------|
| FGSM-AT | FGSM | 低 | 弱 |
| PGD-AT | PGD | 高 | 强 |
| TRADES | PGD+KL | 高 | 最强 |
| FreeAT | FGSM复用 | 低 | 中 |

### 鲁棒性-精度权衡
对抗训练提升鲁棒性但通常降低干净样本精度，这是已知的trade-off。

In [ ]:
class AdversarialTraining:
    def __init__(self, model, epsilon=0.1, alpha=0.01, pgd_steps=5, method='pgd'):
        self.model = model
        self.epsilon = epsilon
        self.alpha = alpha
        self.pgd_steps = pgd_steps
        self.method = method

    def generate_adversarial(self, x, y):
        if self.method == 'fgsm':
            x_adv = x.clone().detach().requires_grad_(True)
            logits = self.model(x_adv)
            loss = F.cross_entropy(logits, y)
            grad = torch.autograd.grad(loss, x_adv)[0]
            return (x + self.epsilon * grad.sign()).detach()
        elif self.method == 'pgd':
            x_adv = x.clone().detach() + torch.zeros_like(x).uniform_(-self.epsilon, self.epsilon)
            for _ in range(self.pgd_steps):
                x_adv.requires_grad_(True)
                logits = self.model(x_adv)
                loss = F.cross_entropy(logits, y)
                grad = torch.autograd.grad(loss, x_adv)[0]
                x_adv = (x_adv.detach() + self.alpha * grad.sign())
                delta = torch.clamp(x_adv - x, -self.epsilon, self.epsilon)
                x_adv = torch.clamp(x + delta, -3, 3)
            return x_adv.detach()
        return x

    def train_step(self, x, y, optimizer, beta=0.5):
        x_adv = self.generate_adversarial(x, y)
        clean_logits = self.model(x)
        adv_logits = self.model(x_adv)
        clean_loss = F.cross_entropy(clean_logits, y)
        adv_loss = F.cross_entropy(adv_logits, y)
        total_loss = (1 - beta) * clean_loss + beta * adv_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        return clean_loss.item(), adv_loss.item()

class TRADESTraining(AdversarialTraining):
    def train_step(self, x, y, optimizer, beta=6.0):
        x_adv = self.generate_adversarial(x, y)
        clean_logits = self.model(x)
        adv_logits = self.model(x_adv)
        ce_loss = F.cross_entropy(clean_logits, y)
        kl_loss = F.kl_div(F.log_softmax(adv_logits, dim=1),
                            F.softmax(clean_logits.detach(), dim=1), reduction='batchmean')
        total_loss = ce_loss + beta * kl_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        return ce_loss.item(), kl_loss.item()

model = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 5))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

at = AdversarialTraining(model, epsilon=0.15, alpha=0.02, pgd_steps=5, method='pgd')
trades = TRADESTraining(model, epsilon=0.15, alpha=0.02, pgd_steps=5, method='pgd')

x = torch.randn(16, 64)
y = torch.randint(0, 5, (16,))

print('=== Adversarial Training ===')
print(f'PGD-AT training:')
for epoch in range(5):
    clean_loss, adv_loss = at.train_step(x, y, optimizer, beta=0.5)
    if (epoch + 1) % 2 == 0:
        print(f'  Epoch {epoch+1}: clean_loss={clean_loss:.4f}, adv_loss={adv_loss:.4f}')

print(f'\nTRADES training:')
for epoch in range(5):
    ce_loss, kl_loss = trades.train_step(x, y, optimizer, beta=6.0)
    if (epoch + 1) % 2 == 0:
        print(f'  Epoch {epoch+1}: ce_loss={ce_loss:.4f}, kl_loss={kl_loss:.4f}')

with torch.no_grad():
    clean_acc = (model(x).argmax(1) == y).float().mean()
    x_adv = at.generate_adversarial(x, y)
    adv_acc = (model(x_adv).argmax(1) == y).float().mean()
print(f'\nFinal: clean_acc={clean_acc:.2%}, adv_acc={adv_acc:.2%}')
print(f'\nKey: Adversarial training trades clean accuracy for robustness.')
print(f'PGD-AT is standard; TRADES explicitly manages the robustness-accuracy trade-off.')

## 4. 安全对齐

**安全对齐**确保模型行为符合人类价值观和安全准则。

### 主要方法
1. **RLHF**：用人类反馈训练奖励模型，再用PPO优化策略
2. **Constitutional AI (CAI)**：用AI反馈替代人类反馈，基于"宪法"原则
3. **DPO**：直接用偏好数据优化策略，无需奖励模型
4. **Red-Team RLHF**：红队攻击发现的漏洞用于增强对齐训练

### CAI流程
1. 有害提示 → 模型生成有害回答
2. 模型自我批评："这个回答有什么问题？"
3. 模型修正：基于批评生成安全回答
4. 用修正后的回答做SFT/RLHF

### 产业实践
- Anthropic: Constitutional AI
- OpenAI: RLHF + Red Teaming
- Google: RLAIF + Safety Layers

In [ ]:
class ConstitutionalAI(nn.Module):
    def __init__(self, d_model=128, n_principles=5):
        super().__init__()
        self.n_principles = n_principles
        self.principle_embeds = nn.Parameter(torch.randn(n_principles, d_model) * 0.1)
        self.critic_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid()
        )
        self.revision_net = nn.Sequential(
            nn.Linear(d_model * 2, 64), nn.ReLU(), nn.Linear(64, d_model)
        )
        self.safety_scorer = nn.Sequential(
            nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid()
        )

    def critique(self, response_embed, principle_idx=0):
        principle = self.principle_embeds[principle_idx].unsqueeze(0).expand(response_embed.size(0), -1)
        critique_input = torch.cat([response_embed, principle], dim=-1)
        violation_score = self.critic_net(critique_input)
        return violation_score

    def revise(self, response_embed, principle_idx=0):
        principle = self.principle_embeds[principle_idx].unsqueeze(0).expand(response_embed.size(0), -1)
        revision_input = torch.cat([response_embed, principle], dim=-1)
        revised = self.revision_net(revision_input)
        return revised

    def full_pipeline(self, initial_response):
        current = initial_response
        for p_idx in range(self.n_principles):
            violation = self.critique(current, p_idx)
            if violation.item() > 0.3:
                current = self.revise(current, p_idx)
        safety_score = self.safety_scorer(current)
        return current, safety_score

cai = ConstitutionalAI(d_model=128, n_principles=5)

harmful_response = torch.randn(1, 128) + 0.5
safe_response = torch.randn(1, 128) - 0.3

print('=== Constitutional AI ===')
print(f'Principle-level critique:')
for i in range(cai.n_principles):
    harm_violation = cai.critique(harmful_response, i)
    safe_violation = cai.critique(safe_response, i)
    print(f'  Principle {i}: harmful={harm_violation.item():.3f}, safe={safe_violation.item():.3f}')

revised, safety = cai.full_pipeline(harmful_response)
print(f'\nFull pipeline: initial_safety={cai.safety_scorer(harmful_response).item():.3f} -> revised_safety={safety.item():.3f}')

print(f'\nKey: CAI iteratively critiques and revises responses against constitutional principles.')
print(f'Each principle catches different types of safety violations.')
print(f'In production, principles are human-written rules like "Be helpful but not harmful".')

## 5. 深度防御与红队测试

### 深度防御 (Defense-in-Depth)

```
用户输入 → [输入过滤] → [输入净化] → [模型推理] → [输出审查] → 安全输出
              ↓              ↓              ↓              ↓
           规则+ML        指令分离       对抗训练       毒性+PII检测
```

### 红队测试

**红队测试**是主动安全评估，模拟攻击者发现系统漏洞：
- **人工红队**：安全专家手动测试
- **自动化红队**：用AI生成攻击
- **众包红队**：邀请外部人员测试

### 安全评估基准
| 基准 | 评估内容 |
|------|----------|
| HarmBench | 标准化危害评估 |
| AdvBench | 对抗鲁棒性 |
| ToxiGen | 毒性检测 |
| TruthfulQA | 事实性 |
| BBQ | 偏见评估 |

In [ ]:
class RedTeamEvaluator:
    def __init__(self, d_model=128, n_attack_types=4):
        self.d_model = d_model
        self.n_attack_types = n_attack_types
        self.attack_names = ['prompt_injection', 'jailbreak', 'gradient_attack', 'social_engineering']
        self.attack_generators = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model))
            for _ in range(n_attack_types)
        ])
        self.defense = SafetyGuardrail(d_model)
        self.results = {name: {'attempts': 0, 'blocked': 0} for name in self.attack_names}

    def run_red_team_test(self, system_embed, n_tests=10):
        for test in range(n_tests):
            attack_type = test % self.n_attack_types
            attack_name = self.attack_names[attack_type]
            base_input = torch.randn(1, self.d_model)
            attack_input = self.attack_generators[attack_type](base_input)
            model_output = torch.randn(1, self.d_model)
            result, status = self.defense.process(system_embed, attack_input, model_output)
            self.results[attack_name]['attempts'] += 1
            if result is None:
                self.results[attack_name]['blocked'] += 1

    def get_report(self):
        report = {}
        for name, data in self.results.items():
            total = data['attempts']
            blocked = data['blocked']
            report[name] = {
                'attempts': total,
                'blocked': blocked,
                'block_rate': blocked / max(total, 1),
                'vulnerability_rate': 1 - blocked / max(total, 1)
            }
        return report

evaluator = RedTeamEvaluator(d_model=128)
system = torch.randn(1, 128)

print('=== Red Team Evaluation ===')
evaluator.run_red_team_test(system, n_tests=20)

report = evaluator.get_report()
print(f'{"Attack Type":<22} {"Attempts":>8} {"Blocked":>8} {"Block Rate":>10} {"Vuln Rate":>10}')
print('-' * 62)
for name, data in report.items():
    print(f'{name:<22} {data["attempts"]:>8} {data["blocked"]:>8} {data["block_rate"]:>10.1%} {data["vulnerability_rate"]:>10.1%}')

total_vuln = sum(d['vulnerability_rate'] for d in report.values()) / len(report)
print(f'\nOverall vulnerability rate: {total_vuln:.1%}')
print(f'\nKey: Red team testing systematically probes all attack surfaces.')
print(f'Low block rates indicate areas needing stronger defenses.')
print(f'In production, red teaming should be continuous, not one-time.')

## 6. 防御方法总结

| 防御层 | 方法 | 防御目标 | 局限性 |
|--------|------|---------|--------|
| 输入过滤 | 规则+ML+困惑度 | 拦截恶意输入 | 可被绕过 |
| 输入净化 | 指令分离+转义 | 减少注入影响 | 可能丢失信息 |
| 对抗训练 | PGD-AT/TRADES | 提升鲁棒性 | 精度-鲁棒性权衡 |
| 安全对齐 | RLHF/CAI/DPO | 行为符合价值观 | 对齐税、越狱风险 |
| 输出审查 | 毒性+PII+事实性 | 拦截有害输出 | 延迟增加 |
| 红队测试 | 主动攻击评估 | 发现漏洞 | 覆盖面有限 |

**核心原则**：
1. **深度防御**：多层防御，无单点故障
2. **持续评估**：红队测试常态化
3. **自适应**：根据新攻击更新防御
4. **最小权限**：模型只做必要的事
5. **可审计**：记录所有安全事件